# DriftSense: Data Generation Pipeline (Phase D1 - D4)

This pipeline synthesizes **DriftBench-v1**:
- **D1**: World demand generation (multi-seasonality, negative-binomial noise, Indian calendar, Poisson deployments) & Infrastructure capacity / autoscaling telemetry across 6 services.
- **D2**: FOCUS pricing model (on-demand, committed, credits/refunds, contracted costs).
- **D3**: Fault injection at root layer (6 fault types, decoys, overlapping, confounded scenarios) & `ground_truth` generation.
- **D4**: Realistic data dirt injection (missing hours, duplicates, late restatements, timezone shifts, unit mix) & `dirt_log`.

### 1. Setup & Environment

In [ ]:
# Mount Drive for Colab persistent artifact storage
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import sys
from pathlib import Path

current_dir = Path.cwd()
if (current_dir / 'src').exists():
    sys.path.insert(0, str(current_dir / 'src'))
elif (current_dir.parent / 'src').exists():
    sys.path.insert(0, str(current_dir.parent / 'src'))

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

import driftsense
from driftsense.sim.world import generate_world_timeline
from driftsense.sim.infra import generate_infra_telemetry
from driftsense.utils.seed import set_seed
from driftsense.utils.io import load_config
from driftsense.utils.summary import print_result_summary

print(f"DriftSense Version: {driftsense.__version__}")

### 2. Configuration & Scale Selection

In [ ]:
import time
start_time = time.time()

base_cfg = load_config("base.yaml")
scale_cfg = load_config("scale.yaml")

# Select scale: 'tiny' for smoke/tests, 'small' for quick Colab verification, 'full' for 365-day benchmark
SCALE = "tiny"  # Change to 'full' for production run in Colab
active_params = scale_cfg[SCALE]

SEED = base_cfg.get("seed", 42)
set_seed(SEED)

DAYS = active_params["days"]
PROVIDERS = base_cfg.get("providers", ["aws_like", "azure_like"])
RES_PER_PROVIDER = active_params["resources_per_provider"]

print(f"Active Scale: {SCALE} | Days: {DAYS} | Providers: {PROVIDERS} | Resources/Provider: {RES_PER_PROVIDER}")

### 3. Run: World & Infrastructure Simulation

In [ ]:
print("Generating world timeline...")
world_df, events_df = generate_world_timeline(
    days=DAYS,
    start_date_str=base_cfg["date_range"]["start"],
    annual_growth_rate=0.35,
    seed=SEED,
)

print("Generating infrastructure telemetry...")
infra_df = generate_infra_telemetry(
    world_df=world_df,
    providers=PROVIDERS,
    resources_per_provider=RES_PER_PROVIDER,
    seed=SEED,
)

print(f"World timeline shape: {world_df.shape}")
print(f"Events logged: {len(events_df)}")
print(f"Infra telemetry shape: {infra_df.shape}")

### 4. Inspect: Telemetry Visualizations & Distributions

In [ ]:
# Plot key simulation dynamics
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=False)

# 1. Traffic Requests over Time
ts_world = world_df["timestamp"].to_numpy()
req_world = world_df["traffic_requests"].to_numpy()
axes[0].plot(ts_world[:min(len(ts_world), 24*14)], req_world[:min(len(req_world), 24*14)], color="royalblue", lw=1.2)
axes[0].set_title("World Web Traffic (Hourly Requests - Diurnal & Weekly Seasonality)", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Requests / hr")
axes[0].grid(True, alpha=0.3)

# 2. CPU vs Requests (Compute Service)
compute_sample = infra_df.filter(pl.col("ServiceName") == "compute").head(24*14)
axes[1].plot(compute_sample["ChargePeriodStart"].to_numpy(), compute_sample["x_cpu_pct"].to_numpy(), color="crimson", label="CPU %")
axes[1].axhline(72.0, color="darkred", linestyle="--", alpha=0.6, label="Autoscale Up Threshold (72%)")
axes[1].axhline(35.0, color="green", linestyle="--", alpha=0.6, label="Autoscale Down Threshold (35%)")
axes[1].set_title("Compute Service CPU Utilization with Autoscaling Bands", fontsize=11, fontweight="bold")
axes[1].set_ylabel("CPU %")
axes[1].legend(loc="upper right")
axes[1].grid(True, alpha=0.3)

# 3. Autoscaled Instance Count
axes[2].step(compute_sample["ChargePeriodStart"].to_numpy(), compute_sample["x_instance_count"].to_numpy(), color="darkorange", where="post", lw=1.5)
axes[2].set_title("Dynamic Instance Count (Autoscaler with Cooldown)", fontsize=11, fontweight="bold")
axes[2].set_ylabel("Instance Count")
axes[2].grid(True, alpha=0.3)

# 4. Cumulative Storage Growth (Object Storage & Database)
storage_sample = infra_df.filter(pl.col("ServiceName") == "object_storage").head(24*14)
axes[3].plot(storage_sample["ChargePeriodStart"].to_numpy(), storage_sample["x_storage_gb"].to_numpy(), color="purple", lw=1.5)
axes[3].set_title("Monotonically Cumulative Object Storage (GB)", fontsize=11, fontweight="bold")
axes[3].set_ylabel("Storage (GB)")
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 5. Checks: Acceptance Criteria Verification
- Zero NaNs in any column
- Monotone storage growth
- Instance counts within bounds
- Autocorrelation peaks at 24h and 168h

In [ ]:
checks_status = {}

# 1. Zero NaNs
nan_violations = {col: infra_df[col].null_count() for col in infra_df.columns if infra_df[col].null_count() > 0}
assert len(nan_violations) == 0, f"NaNs found: {nan_violations}"
checks_status["zero_nans"] = "PASS"

# 2. Instance count bounds
assert (infra_df["x_instance_count"] >= 1).all()
checks_status["instance_counts_bounded"] = "PASS"

# 3. Storage monotonicity
storage_res = infra_df.filter(pl.col("ServiceName") == "object_storage")["ResourceId"].unique().to_list()[0]
storage_arr = infra_df.filter(pl.col("ResourceId") == storage_res)["x_storage_gb"].to_numpy()
storage_diff = np.diff(storage_arr)
assert (storage_diff >= -1e-6).all(), f"Storage decreased: min delta={storage_diff.min()}"
checks_status["storage_monotone"] = "PASS"

# 4. Autocorrelation (ACF) peaks
from statsmodels.tsa.stattools import acf
req_acf = acf(world_df["traffic_requests"].to_numpy(), nlags=min(len(world_df)-1, 180))
acf_24 = req_acf[24] if len(req_acf) > 24 else 1.0
acf_168 = req_acf[168] if len(req_acf) > 168 else 1.0
assert acf_24 > 0.25, f"24h ACF peak weak: {acf_24}"
checks_status["acf_24h_peak"] = "PASS"

print("Acceptance Checks Results:", checks_status)

### 6. Save Artifacts

In [ ]:
out_dir = Path("artifacts/d1")
out_dir.mkdir(parents=True, exist_ok=True)

world_path = out_dir / "world_timeline.parquet"
events_path = out_dir / "events.parquet"
infra_path = out_dir / "infra_telemetry.parquet"

world_df.write_parquet(world_path)
events_df.write_parquet(events_path)
infra_df.write_parquet(infra_path)

artifacts_saved = [str(world_path), str(events_path), str(infra_path)]
print("Saved artifacts:", artifacts_saved)

### 7. RESULT SUMMARY

In [ ]:
runtime = time.time() - start_time

summary = {
    "phase": "D1",
    "scale": SCALE,
    "seed": SEED,
    "runtime_sec": runtime,
    "shapes": {
        "world_df": f"{world_df.height} x {world_df.width}",
        "events_df": f"{events_df.height} x {events_df.width}",
        "infra_df": f"{infra_df.height} x {infra_df.width}",
    },
    "key_metrics": {
        "mean_hourly_requests": float(round(world_df["traffic_requests"].mean(), 2)),
        "acf_24h": float(round(acf_24, 4)),
        "total_events_logged": len(events_df),
    },
    "checks": checks_status,
    "warnings": [],
    "artifacts": artifacts_saved
}

print_result_summary(summary)